# ScholarMind Demo Notebook

This notebook demonstrates the core functionality of the ScholarMind AI Research Assistant.

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import asyncio
import pandas as pd
import matplotlib.pyplot as plt

print('🚀 ScholarMind Demo Ready!')

## 1. Document Processing Demo

In [ ]:
from backend.core.document_processing import DocumentChunker, MetadataExtractor

# Sample research paper text
sample_paper = """
Title: Advances in Knowledge Graph Construction for Scientific Research

Abstract:
Knowledge graphs have emerged as a powerful tool for organizing and querying 
scientific information. This paper presents a novel approach to automatically 
constructing knowledge graphs from research papers using advanced NLP techniques.

Introduction:
The exponential growth of scientific literature has made it increasingly difficult 
for researchers to keep up with advances in their fields. Knowledge graphs offer 
a solution by representing information as interconnected entities and relationships.

Methods:
We employ a pipeline consisting of named entity recognition, relation extraction, 
and entity linking. Our system uses transformer-based models fine-tuned on 
scientific text.

Results:
Our approach achieves 85% precision and 78% recall on entity extraction, and 
72% accuracy on relation classification. The resulting knowledge graph contains 
over 50,000 entities and 120,000 relationships.

Conclusion:
Knowledge graphs constructed from scientific literature can significantly 
accelerate research by enabling semantic search and automated hypothesis generation.
"""

# Extract metadata
metadata_extractor = MetadataExtractor()
metadata = metadata_extractor.extract(sample_paper, {})

print('📄 Extracted Metadata:')
print(f'   Title: {metadata.title}')
print(f'   Abstract: {metadata.abstract[:100]}...')
print(f'   Keywords: {metadata.keywords}')

In [ ]:
# Chunk the document
chunker = DocumentChunker(chunk_size=256, chunk_overlap=25)
chunks = chunker.chunk(sample_paper, 'demo-paper')

print(f'📦 Created {len(chunks)} chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f'\n   Chunk {i+1} ({len(chunk.text)} chars):')
    print(f'   "{chunk.text[:80]}..."')

## 2. Entity & Relation Extraction Demo

In [ ]:
from backend.core.nlp import EntityExtractor, RelationExtractor

# Extract entities
entity_extractor = EntityExtractor(use_scispacy=False)
entities = entity_extractor.extract(sample_paper, 'demo-paper')

print(f'🔍 Extracted {len(entities)} entities:')
for entity in entities[:10]:
    print(f'   - {entity.text} ({entity.entity_type}, conf: {entity.confidence:.2f})')

In [ ]:
# Extract relations
relation_extractor = RelationExtractor()
entity_list = [{'text': e.text, 'entity_type': e.entity_type} for e in entities]
relations = relation_extractor.extract(sample_paper, entity_list)

print(f'🔗 Extracted {len(relations)} relations:')
for rel in relations[:5]:
    print(f'   - {rel.source_text} --[{rel.relation_type}]--> {rel.target_text}')

## 3. Knowledge Graph Construction Demo

In [ ]:
from backend.core.knowledge_graph import GraphBuilder, CommunityDetector
import networkx as nx

# Build knowledge graph
graph_builder = GraphBuilder()

entity_dicts = [{'text': e.text, 'entity_type': e.entity_type, 'confidence': e.confidence} for e in entities]
relation_dicts = [{'source_text': r.source_text, 'target_text': r.target_text, 
                   'relation_type': r.relation_type, 'confidence': r.confidence} for r in relations]

graph_builder.build_from_extraction(entity_dicts, relation_dicts)

stats = graph_builder.get_statistics()
print(f'📊 Knowledge Graph Statistics:')
print(f'   Nodes: {stats["nodes"]}')
print(f'   Edges: {stats["edges"]}')
print(f'   Density: {stats["density"]:.4f}')

In [ ]:
# Visualize the graph
G = graph_builder.graph

if G.number_of_nodes() > 0:
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    nx.draw(G, pos, 
            node_color='lightblue', 
            node_size=500,
            with_labels=True, 
            font_size=8,
            arrows=True,
            arrowsize=10)
    
    plt.title('Knowledge Graph from Research Paper')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Graph is empty - add more text for entity extraction')

## 4. Semantic Search Demo

In [ ]:
from backend.core.nlp import EmbeddingsManager
from backend.core.retrieval import VectorStore

# Initialize embedding manager and vector store
embeddings_manager = EmbeddingsManager()
vector_store = VectorStore(collection_name='demo-collection')

# Index document chunks
chunk_texts = [chunk.text for chunk in chunks]
chunk_embeddings = embeddings_manager.embed_documents(chunk_texts)

vector_store.add(
    texts=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[{'chunk_id': i, 'document_id': 'demo-paper'} for i in range(len(chunks))],
)

print(f'✅ Indexed {len(chunks)} chunks in vector store')

In [ ]:
# Search for relevant content
query = "What are the results of the knowledge graph construction?"

query_embedding = embeddings_manager.embed_query(query)
results = vector_store.search(query_embedding, k=3)

print(f'🔎 Search Results for: "{query}"\n')
for i, result in enumerate(results):
    print(f'Result {i+1} (score: {result["score"]:.4f}):')
    print(f'   {result["text"][:150]}...')
    print()

## 5. Research Agent Demo

In [ ]:
from backend.core.agents import ResearchAgent

# Note: This requires API key configuration
# Initialize research agent
try:
    agent = ResearchAgent()
    print('🤖 Research Agent initialized!')
    print('   The agent can answer research questions using the knowledge graph and documents.')
except Exception as e:
    print(f'⚠️ Agent initialization requires API key: {e}')

In [ ]:
# Example agent query (requires API key)
# result = await agent.query("What are the main components of knowledge graph construction?")
# print(result)

print('💡 To use the Research Agent:')
print('   1. Set GOOGLE_API_KEY in your .env file')
print('   2. Run: result = await agent.query("your question")')

## 6. Summary

This demo showcased the core capabilities of ScholarMind:

1. **Document Processing**: Parse and chunk research documents
2. **Entity Extraction**: Identify key concepts, people, and organizations
3. **Relation Extraction**: Discover relationships between entities
4. **Knowledge Graph**: Build structured representations of knowledge
5. **Semantic Search**: Find relevant information using natural language
6. **Research Agent**: AI-powered question answering over your documents

For more information, see the project README and documentation.

In [ ]:
print('\n🎉 Demo Complete!')
print('\nNext Steps:')
print('  1. Upload your own research documents')
print('  2. Run the full processing pipeline')
print('  3. Explore the knowledge graph in the web interface')
print('  4. Ask questions using the chat interface')